# Silver Layer Transformation: Conformed Dimensions & Facts
**Framework:** Metadata-Driven Pipeline Framework  
**Source Catalog:** `bronze_catalog.raw_data`  
**Target Catalog:** `silver_catalog.insurance_conformed`  
**Capabilities:**  
1. Data hygiene, string trimming, case standardization, null handling.
2. Strict data typing (cast to Decimal, Date, Timestamp, Integer).
3. Conformed Dimensions with SCD Type 1 Upsert (`dim_customer`, `dim_agent`, `dim_coverage`).
4. Enriched Fact Tables with derived business indicators (`fct_policy`, `fct_claim`).
5. Delta Lake `MERGE WITH SCHEMA EVOLUTION` and audit timestamps (`SilverIngestedAt`).

In [ ]:
# 1. Input Widgets
dbutils.widgets.text("bronze_catalog", "bronze_catalog", "Bronze Catalog")
dbutils.widgets.text("bronze_schema", "raw_data", "Bronze Schema")
dbutils.widgets.text("silver_catalog", "silver_catalog", "Silver Catalog")
dbutils.widgets.text("silver_schema", "insurance_conformed", "Silver Schema")

bronze_catalog = dbutils.widgets.get("bronze_catalog").strip()
bronze_schema  = dbutils.widgets.get("bronze_schema").strip()
silver_catalog = dbutils.widgets.get("silver_catalog").strip()
silver_schema  = dbutils.widgets.get("silver_schema").strip()

print(f"Source Bronze: `{bronze_catalog}`.`{bronze_schema}`")
print(f"Target Silver: `{silver_catalog}`.`{silver_schema}`")

# Ensure Silver Schema Exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{silver_catalog}`.`{silver_schema}`")
print(f"Verified schema `{silver_catalog}`.`{silver_schema}`.")

In [ ]:
# 2. Transform Dimension: dim_customer (SCD Type 1)
from pyspark.sql.functions import col, current_timestamp, initcap, lower, trim, concat_ws, coalesce, lit, to_date, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DecimalType, DateType

print("Transforming Customers -> dim_customer...")
try:
    df_cust_raw = spark.table(f"`{bronze_catalog}`.`{bronze_schema}`.`Customers`")
    
    # Standardize casing, trim whitespaces, compute FullName
    df_cust_clean = df_cust_raw \
        .withColumn("CustomerID", col("CustomerID").cast(IntegerType())) \
        .withColumn("FirstName", initcap(trim(col("FirstName")))) \
        .withColumn("LastName", initcap(trim(col("LastName")))) \
        .withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName"))) \
        .withColumn("Email", lower(trim(col("Email")))) \
        .withColumn("Phone", trim(col("Phone"))) \
        .withColumn("City", initcap(trim(col("City")))) \
        .withColumn("State", trim(col("State"))) \
        .withColumn("ZipCode", trim(col("ZipCode"))) \
        .withColumn("SilverIngestedAt", current_timestamp())
        
    # Deduplicate by CustomerID keeping latest ingested
    w_cust = Window.partitionBy("CustomerID").orderBy(col("_ingested_at").desc())
    df_cust_dedup = df_cust_clean.withColumn("_rn", row_number().over(w_cust)).filter(col("_rn") == 1).drop("_rn")
    
    # Write / Merge into Silver dim_customer
    target_cust = f"`{silver_catalog}`.`{silver_schema}`.`dim_customer`"
    df_cust_dedup.createOrReplaceTempView("src_cust")
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_cust}
    USING DELTA
    AS SELECT * FROM src_cust WHERE 1=0
    """)
    
    spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {target_cust} AS target
    USING src_cust AS source
    ON target.CustomerID = source.CustomerID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Successfully synced dim_customer ({df_cust_dedup.count()} records).")
except Exception as e_cust:
    print(f"Note on dim_customer: {str(e_cust)}")

In [ ]:
# 3. Transform Dimension: dim_agent (SCD Type 1)
print("Transforming Agents -> dim_agent...")
try:
    df_agent_raw = spark.table(f"`{bronze_catalog}`.`{bronze_schema}`.`Agents`")
    
    df_agent_clean = df_agent_raw \
        .withColumn("AgentID", col("AgentID").cast(IntegerType())) \
        .withColumn("FirstName", initcap(trim(col("FirstName")))) \
        .withColumn("LastName", initcap(trim(col("LastName")))) \
        .withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName"))) \
        .withColumn("Email", lower(trim(col("Email")))) \
        .withColumn("Region", initcap(trim(col("Region")))) \
        .withColumn("BranchName", trim(col("BranchName"))) \
        .withColumn("SilverIngestedAt", current_timestamp())
        
    target_agent = f"`{silver_catalog}`.`{silver_schema}`.`dim_agent`"
    df_agent_clean.createOrReplaceTempView("src_agent")
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_agent}
    USING DELTA
    AS SELECT * FROM src_agent WHERE 1=0
    """)
    
    spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {target_agent} AS target
    USING src_agent AS source
    ON target.AgentID = source.AgentID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Successfully synced dim_agent ({df_agent_clean.count()} records).")
except Exception as e_agent:
    print(f"Note on dim_agent: {str(e_agent)}")

In [ ]:
# 4. Transform Dimension: dim_coverage (Reference / Dimension)
print("Transforming CoverageTypes -> dim_coverage...")
try:
    df_cov_raw = spark.table(f"`{bronze_catalog}`.`{bronze_schema}`.`CoverageTypes`")
    
    df_cov_clean = df_cov_raw \
        .withColumn("CoverageCode", trim(col("CoverageCode"))) \
        .withColumn("CoverageName", trim(col("CoverageName"))) \
        .withColumn("Category", initcap(trim(coalesce(col("Category"), lit("General"))))) \
        .withColumn("Description", trim(coalesce(col("Description"), col("CoverageName")))) \
        .withColumn("SilverIngestedAt", current_timestamp())
        
    target_cov = f"`{silver_catalog}`.`{silver_schema}`.`dim_coverage`"
    df_cov_clean.createOrReplaceTempView("src_cov")
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_cov}
    USING DELTA
    AS SELECT * FROM src_cov WHERE 1=0
    """)
    
    spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {target_cov} AS target
    USING src_cov AS source
    ON target.CoverageCode = source.CoverageCode
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Successfully synced dim_coverage ({df_cov_clean.count()} records).")
except Exception as e_cov:
    print(f"Note on dim_coverage: {str(e_cov)}")

In [ ]:
# 5. Transform Fact: fct_policy (Conformed Policy Fact)
from pyspark.sql.functions import datediff, when

print("Transforming Policies -> fct_policy...")
try:
    df_pol_raw = spark.table(f"`{bronze_catalog}`.`{bronze_schema}`.`Policies`")
    
    df_pol_clean = df_pol_raw \
        .withColumn("PolicyID", col("PolicyID").cast(IntegerType())) \
        .withColumn("CustomerID", col("CustomerID").cast(IntegerType())) \
        .withColumn("AgentID", col("AgentID").cast(IntegerType())) \
        .withColumn("CoverageCode", trim(col("CoverageCode"))) \
        .withColumn("PolicyStartDate", to_date(col("PolicyStartDate"))) \
        .withColumn("PolicyEndDate", to_date(col("PolicyEndDate"))) \
        .withColumn("PremiumAmount", coalesce(col("PremiumAmount").cast(DecimalType(12,2)), lit(0.00))) \
        .withColumn("DeductibleAmount", coalesce(col("DeductibleAmount").cast(DecimalType(12,2)), lit(0.00))) \
        .withColumn("PolicyStatus", trim(col("PolicyStatus"))) \
        .withColumn("PolicyTenureDays", datediff(col("PolicyEndDate"), col("PolicyStartDate"))) \
        .withColumn("IsActivePolicy", when(col("PolicyStatus").isin("ACTIVE", "Active"), lit(True)).otherwise(lit(False))) \
        .withColumn("SilverIngestedAt", current_timestamp())
        
    target_pol = f"`{silver_catalog}`.`{silver_schema}`.`fct_policy`"
    df_pol_clean.createOrReplaceTempView("src_pol")
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_pol}
    USING DELTA
    AS SELECT * FROM src_pol WHERE 1=0
    """)
    
    spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {target_pol} AS target
    USING src_pol AS source
    ON target.PolicyID = source.PolicyID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Successfully synced fct_policy ({df_pol_clean.count()} records).")
except Exception as e_pol:
    print(f"Note on fct_policy: {str(e_pol)}")

In [ ]:
# 6. Transform Fact: fct_claim (Conformed Claims Fact)
print("Transforming Claims -> fct_claim...")
try:
    df_claim_raw = spark.table(f"`{bronze_catalog}`.`{bronze_schema}`.`Claims`")
    
    df_claim_clean = df_claim_raw \
        .withColumn("ClaimID", col("ClaimID").cast(IntegerType())) \
        .withColumn("PolicyID", col("PolicyID").cast(IntegerType())) \
        .withColumn("ClaimDate", to_date(col("ClaimDate"))) \
        .withColumn("IncidentDate", to_date(col("IncidentDate"))) \
        .withColumn("ClaimAmount", coalesce(col("ClaimAmount").cast(DecimalType(12,2)), lit(0.00))) \
        .withColumn("SettledAmount", coalesce(col("SettledAmount").cast(DecimalType(12,2)), lit(0.00))) \
        .withColumn("ClaimStatus", trim(col("ClaimStatus"))) \
        .withColumn("SettlementDays", datediff(col("ClaimDate"), col("IncidentDate"))) \
        .withColumn("SilverIngestedAt", current_timestamp())
        
    target_claim = f"`{silver_catalog}`.`{silver_schema}`.`fct_claim`"
    df_claim_clean.createOrReplaceTempView("src_claim")
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_claim}
    USING DELTA
    AS SELECT * FROM src_claim WHERE 1=0
    """)
    
    spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {target_claim} AS target
    USING src_claim AS source
    ON target.ClaimID = source.ClaimID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Successfully synced fct_claim ({df_claim_clean.count()} records).")
except Exception as e_claim:
    print(f"Note on fct_claim: {str(e_claim)}")

In [ ]:
# 7. Return Execution Output JSON
import json

output = {
    "status": "Success",
    "layer": "Silver",
    "target_schema": f"{silver_catalog}.{silver_schema}",
    "dimensions_updated": ["dim_customer", "dim_agent", "dim_coverage"],
    "facts_updated": ["fct_policy", "fct_claim"]
}

print(json.dumps(output, indent=2))
dbutils.notebook.exit(json.dumps(output))